In [ ]:
import numpy as np
import pandas as pd

import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.cluster import KMeans


import joblib

In [ ]:
df = pd.read_csv(r'C:\Users\kulde\kuldeepcode\project\customer_dataset.csv')

#### Data Cleaning ( EDA )

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.duplicated().sum()

In [ ]:
df.isnull().sum()

In [ ]:
df['Age'] = df['Age'].fillna(df['Age'].median())
df['Income'] = df['Income'].fillna(df['Income'].median())

In [ ]:
df['SpendingScore'] = df['SpendingScore'].fillna(df['SpendingScore'].median())

In [ ]:
df = df.drop_duplicates()

In [ ]:
df.isna().sum()

In [ ]:
df.isna().any().sum()

In [ ]:
df['Age'] = df['Age'].astype('int')
df['Income'] = df['Income'].astype('int')
df['SpendingScore'] = df['SpendingScore'].astype('int')

In [ ]:
df.info()

In [ ]:
df.head()

In [ ]:
df['City'].unique().value_counts().sum()
df['City'].value_counts()

In [ ]:
df['ProductCategory'].unique().value_counts().sum()

df['ProductCategory'].value_counts()

In [ ]:
df['MembershipStatus'].unique().value_counts().sum()

df['MembershipStatus'].value_counts()

In [ ]:
dfx = df.drop(columns=['CustomerID'])
dfx

#### Pipline Create and Upcoming Data Handling 

In [ ]:
num_value = dfx.select_dtypes(include='number').columns
cat_value = dfx.select_dtypes(include=['object', 'category', 'string']).columns

In [ ]:
preprocess = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_value),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_value)
    ]
)

In [ ]:
model = Pipeline(steps=[
    ('preprocess', preprocess),
    ('KMeans', KMeans(n_clusters=4, random_state=42, n_init=10))
])

#### Create KMeans Cluster Data 

In [ ]:
model.fit(dfx)
clusters = model.predict(dfx)
dfx['clusters'] = clusters

In [ ]:
dfx['clusters'].value_counts()

In [ ]:
dfx.groupby('clusters').mean(numeric_only=True)

In [ ]:
cluster_names = {
    0: "Low Value Inactive Customers",
    1: "Occasional Regular Customers",
    2: "High Income Underutilized VIPs",
    3: "Active High Value Customers"
}

dfx['cluster_name'] = dfx['clusters'].map(cluster_names)

In [ ]:
dfx[['clusters', 'cluster_name']].head()

In [ ]:
dfx['cluster_name'].value_counts()

In [ ]:
dfx.groupby('cluster_name').mean(numeric_only=True)

In [ ]:
sns.scatterplot(
    data=dfx,
    x='Income',
    y='SpendingScore',
    hue='cluster_name'
)

plt.show()

In [ ]:
sns.boxplot(data=dfx, x='cluster_name', y='SpendingScore')
plt.xticks(rotation=45)
plt.show()

In [ ]:
sns.scatterplot(
    data=dfx,
    x='SpendingScore',
    y='Income',
    hue='cluster_name'
)

plt.show()

#### Save Model


In [ ]:
joblib.dump(model, "kmeans_pipeline.pkl")